# TransE, TransH y TransR

Solo estos tres modelos y solo el grafo de SteelVault, el más pequeño y el único con las partes
verificadas a mano. Cinco pasos:

1. el grafo como tripletas `(h, r, t)`;
2. qué hace cada modelo;
3. entrenarlos y ver qué han aprendido;
4. ¿adivinan aristas que no han visto?;
5. la pregunta que nos importa: ¿adivinan la parte de un enunciado?, y por qué no.

In [ ]:
import collections
import sys
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import Markdown, display

for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"):
    if (folder / "link_prediction.py").exists():
        sys.path.insert(0, str(folder))
        break
import link_prediction as lp

# Configuración de cada modelo: la que eligió el ajuste de link_prediction.ipynb (sección 3).
#   loss           "1vsall": cada tripleta verdadera contra las 183 entidades a la vez
#                  "margin": contra 32 tripletas falsas hechas al azar (pérdida de margen, la de los artículos)
#   lr             tasa de aprendizaje: cuánto se mueven los vectores en cada corrección
#   epochs         cuántas veces se recorren todas las tripletas
#   embedding_dim  números por entidad; TransR usa la mitad (16) en el espacio de cada relación
#   batch_size     tripletas por corrección
#   negatives      tripletas falsas por cada verdadera (solo con "margin")
# El optimizador es Adam en los tres.
CONFIG = {
    "TransE": {"loss": "1vsall", "lr": 0.03, "epochs": 100, "embedding_dim": 128, "batch_size": 256, "negatives": 32},
    "TransH": {"loss": "margin", "lr": 0.1, "epochs": 300, "embedding_dim": 32, "batch_size": 256, "negatives": 32},
    "TransR": {"loss": "margin", "lr": 0.03, "epochs": 100, "embedding_dim": 32, "batch_size": 256, "negatives": 32},
}
MODELS = tuple(CONFIG)

contract_name = "SteelVault"
SV = lp.load_contract(contract_name)
e2i, r2i = lp.entity_ids(SV)
i2e = {i: e for e, i in e2i.items()}
print("dispositivo:", lp.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

dispositivo: cuda | Quadro RTX 5000


In [3]:
print(f"contrato {contract_name}")
print(f"{len(SV.triples)} tripletas, {len(e2i)} entidades, {len(r2i)} tipos de arista presentes")
lp.table([{"tipo de arista": r, "tripletas": n} for r, n in collections.Counter(r for _, r, _ in SV.triples).most_common()])

contrato SteelVault
275 tripletas, 183 entidades, 7 tipos de arista presentes


| tipo de arista | tripletas |
|---|---|
| is_part_of | 156 |
| assigns_obligation_to | 65 |
| uses | 21 |
| grants_right_to | 18 |
| defines | 8 |
| modifies | 6 |
| references | 1 |

En SteelVault no aparecen `depends_on` ni `supersedes`: la extracción no produjo ninguna.

Algunas tripletas, con el texto de cada nodo para poder leerlas:

In [4]:
def label(node):
    if node in SV.parties:
        return f"{node} ({SV.parties[node]['name']})"
    if node in SV.clauses:
        c = SV.clauses[node]
        return f"{node} (§{c['ref']} {c.get('heading') or ''})".strip()
    if node in SV.statements:
        return f"{node} «{lp.statement_text(SV.statements[node])[:55]}…»"
    return node


examples = [("obligation-3", "assigns_obligation_to", "party-2"), ("obligation-3", "is_part_of", "clause-1"),
            ("right-3", "grants_right_to", "party-1"), ("condition-3", "is_part_of", "obligation-7"),
            ("right-15", "modifies", "clause-22")]
assert all(t in set(SV.triples) for t in examples)
lp.table([{"h": label(h), "r": r, "t": label(t)} for h, r, t in examples])

| h | r | t |
|---|---|---|
| obligation-3 «Marketing Affiliate shall pay to Equidata, as compensat…» | assigns_obligation_to | party-2 (National Credit Report.com, LLC) |
| obligation-3 «Marketing Affiliate shall pay to Equidata, as compensat…» | is_part_of | clause-1 (§1 Compensation) |
| right-3 «Equidata reserves the right to increase the base cost o…» | grants_right_to | party-1 (Equidata, Inc.) |
| condition-3 | is_part_of | obligation-7 «Marketing Affiliate agrees to reimburse Equidata all co…» |
| right-15 «EQUIDATA EXPRESSLY DISCLAIMS ANY WARRANTY OF ANY KIND W…» | modifies | clause-22 (§8.4 ) |

## 2. Qué hace cada modelo

Los tres dan a cada entidad un vector de números (64 o 128, según la configuración) y a cada
tipo de arista otro vector. Una tripleta es «verdadera» si sus vectores encajan de una cierta
forma; cada modelo define esa forma:

| Modelo | Una tripleta es verdadera si… | Puntuación (más alta = más creíble) |
|---|---|---|
| **TransE** | origen + relación ≈ destino | `−‖h + r − t‖` |
| **TransH** | lo mismo, pero después de proyectar origen y destino sobre un **plano propio de cada relación** | `−‖h⊥ + r − t⊥‖`, con `h⊥ = h − (wᵣ·h) wᵣ` |
| **TransR** | lo mismo, pero después de llevar origen y destino a **un espacio propio de cada relación** con una matriz | `−‖Mᵣh + r − Mᵣt‖` |

**El problema del muchos-a-uno**, con los datos de SteelVault: 40 aristas `assigns_obligation_to`
apuntan a la misma parte, `party-2`. TransE quiere que, para las 40, `h + r ≈ party-2`, así que
empuja a los 40 enunciados hacia el mismo punto, `party-2 − r`. Pero esos enunciados también
tienen que estar en sitios distintos para sus otras aristas (cada uno en su cláusula). TransH y
TransR nacieron para esto: solo exigen que coincidan **en la proyección** de esa relación, y fuera
de ella cada enunciado puede estar donde le haga falta.

## 3. Entrenar y ver qué han aprendido

**Entrenar** es mover los vectores para que las tripletas verdaderas puntúen más que las falsas.
Para cada tripleta conocida se le pregunta al modelo `(h, r, ?)`: se puntúan las 183 entidades
como posible destino, y se corrigen los vectores para que la verdadera quede arriba. Lo mismo
con el origen, `(?, r, t)`. Se repite unas cien o trescientas veces sobre todas las tripletas
(las *épocas*).

In [5]:
models = {}
rows = []
for m in MODELS:
    start = time.time()
    models[m] = lp.train_kge(m, SV.triples, e2i, r2i, seed=0, **CONFIG[m])
    rows.append({"modelo": m, **CONFIG[m], "segundos": round(time.time() - start, 1)})
lp.table(rows, ["modelo", "epochs", "embedding_dim", "lr", "loss", "segundos"])

/home/sante/Documents/FGV/me/Contract-Knowledge-Graph/server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| modelo | epochs | embedding_dim | lr | loss | segundos |
|---|---|---|---|---|---|
| TransE | 100 | 128 | 0.030 | 1vsall | 3.800 |
| TransH | 300 | 32 | 0.100 | margin | 10.300 |
| TransR | 100 | 32 | 0.030 | margin | 3.400 |

**Cómo se lee un modelo.** 

Se le hace una pregunta, por ejemplo `(obligation-3,
assigns_obligation_to, ?)` —¿a quién obliga *«Marketing Affiliate shall pay to Equidata…»*?—. El
modelo puntúa las 183 entidades y las ordena. La posición de la respuesta correcta es su
**rango**: 1 si sale la primera.

Las cinco primeras de cada modelo. La respuesta correcta es `party-2`, y el modelo la ha visto al
entrenar: esto mide memoria, no deducción.

In [6]:
def top(model, h, r, k=5):
    scores = lp.kge_tail_scores(model, [(h, r)], e2i, r2i)[0]
    order = np.argsort(-scores)
    return [f"{i2e[i]} ({scores[i]:.2f})" for i in order[:k]]


lp.table([{"modelo": m, **{f"{k + 1}.º": x for k, x in enumerate(top(models[m], "obligation-3", "assigns_obligation_to"))}} for m in MODELS])

| modelo | 1.º | 2.º | 3.º | 4.º | 5.º |
|---|---|---|---|---|---|
| TransE | party-2 (-22.34) | party-1 (-24.46) | obligation-3 (-27.47) | reference-1 (-27.70) | condition-8 (-27.84) |
| TransH | obligation-3 (-1.00) | party-2 (-1.08) | obligation-39 (-1.28) | obligation-10 (-1.37) | obligation-7 (-1.37) |
| TransR | party-2 (-0.24) | obligation-39 (-0.71) | obligation-17 (-0.77) | obligation-48 (-0.87) | obligation-42 (-0.88) |

Tiene el formato de Entidad (puntuacion)

## 4. ¿Adivinan aristas que no han visto?

Ahora la prueba de verdad: se esconde un 10% de las tripletas al azar (27), se entrena con el
resto y se pregunta por las escondidas. Para resumir 27 rangos en un número se usan dos medidas:

- **MRR** (*rango recíproco medio*): la media de `1 / rango`. Si la respuesta sale 1.ª, 2.ª y
  4.ª en tres preguntas, el MRR es `(1 + 1/2 + 1/4) / 3 = 0,58`. Vale 1 si siempre sale la primera.
- **Hits@k**: la fracción de preguntas en que la respuesta sale entre las *k* primeras.

El rango es **filtrado**: si una pregunta tiene varias respuestas verdaderas conocidas (una
cláusula tiene muchos enunciados), las otras no cuentan como error. Como referencia, el azar:
ordenar las 183 entidades sin mirar nada.

In [7]:
rng = np.random.default_rng(0)
hidden_idx = set(rng.choice(len(SV.triples), size=len(SV.triples) // 10, replace=False).tolist())
train = [t for i, t in enumerate(SV.triples) if i not in hidden_idx]
hidden = [SV.triples[i] for i in sorted(hidden_idx)]
true_tails = collections.defaultdict(set)
for h, r, t in SV.triples:
    true_tails[(h, r)].add(e2i[t])


def filtered_ranks(model, triples):
    scores = lp.kge_tail_scores(model, [(h, r) for h, r, _ in triples], e2i, r2i)
    ranks = []
    for row, (h, r, t) in zip(scores, triples):
        row = row.copy()
        row[[i for i in true_tails[(h, r)] if i != e2i[t]]] = -np.inf
        ranks.append(1 + int(np.sum(row > row[e2i[t]])))
    return np.array(ranks)


n = len(e2i)
rows = [{"modelo": "al azar (esperado)", "MRR": float(np.mean([1 / k for k in range(1, n + 1)])), "Hits@1": 1 / n, "Hits@10": 10 / n}]
ranks_by_model = {}
for m in MODELS:
    ranks = filtered_ranks(lp.train_kge(m, train, e2i, r2i, seed=0, **CONFIG[m]), hidden)
    ranks_by_model[m] = ranks
    rows.append({"modelo": m, "MRR": float(np.mean(1 / ranks)), "Hits@1": float(np.mean(ranks == 1)), "Hits@10": float(np.mean(ranks <= 10))})
display(Markdown(f"{len(hidden)} tripletas escondidas"))
lp.table(rows)

27 tripletas escondidas

| modelo | MRR | Hits@1 | Hits@10 |
|---|---|---|---|
| al azar (esperado) | 0.032 | 0.005 | 0.055 |
| TransE | 0.490 | 0.333 | 0.741 |
| TransH | 0.182 | 0.074 | 0.407 |
| TransR | 0.166 | 0.148 | 0.185 |

Como referencia, en el ajuste de [`link_prediction.ipynb`](link_prediction.ipynb) —otra muestra
del 10% y dos semillas— el MRR fue 0,46 para TransE, 0,23 para TransH y 0,11 para TransR.

El rango de cada tripleta escondida, para ver en qué tipo de arista acierta cada modelo:

In [8]:
lp.table([{"h": h, "r": r, "t": t, **{m: int(ranks_by_model[m][k]) for m in MODELS}} for k, (h, r, t) in enumerate(hidden)])

| h | r | t | TransE | TransH | TransR |
|---|---|---|---|---|---|
| clause-1 | defines | term-3 | 1 | 49 | 1 |
| clause-13 | uses | term-1 | 18 | 37 | 72 |
| clause-19 | is_part_of | clause-18 | 7 | 6 | 67 |
| clause-5 | defines | term-5 | 1 | 1 | 1 |
| condition-24 | is_part_of | right-12 | 63 | 65 | 57 |
| obligation-14 | assigns_obligation_to | party-2 | 1 | 128 | 158 |
| obligation-17 | is_part_of | clause-2 | 2 | 4 | 151 |
| obligation-19 | is_part_of | clause-2 | 3 | 5 | 81 |
| obligation-34 | assigns_obligation_to | party-1 | 2 | 133 | 162 |
| obligation-43 | is_part_of | clause-21 | 70 | 108 | 161 |
| obligation-44 | uses | term-2 | 1 | 1 | 103 |
| obligation-49 | is_part_of | clause-26 | 14 | 5 | 76 |
| obligation-51 | assigns_obligation_to | party-1 | 2 | 162 | 162 |
| obligation-52 | is_part_of | clause-26 | 8 | 2 | 116 |
| obligation-8 | assigns_obligation_to | party-1 | 2 | 154 | 162 |
| obligation-8 | is_part_of | clause-1 | 5 | 8 | 5 |
| prohibition-11 | assigns_obligation_to | party-1 | 2 | 154 | 117 |
| prohibition-11 | is_part_of | clause-8 | 11 | 115 | 110 |
| prohibition-17 | is_part_of | clause-24 | 1 | 3 | 1 |
| prohibition-3 | assigns_obligation_to | party-2 | 1 | 122 | 183 |
| reference-3 | is_part_of | obligation-12 | 72 | 85 | 54 |
| reference-5 | is_part_of | clause-5 | 2 | 17 | 35 |
| right-11 | grants_right_to | party-1 | 1 | 44 | 170 |
| right-16 | grants_right_to | party-1 | 1 | 139 | 166 |
| right-17 | is_part_of | clause-26 | 24 | 3 | 109 |
| right-6 | is_part_of | clause-12 | 8 | 11 | 1 |
| right-8 | modifies | clause-14 | 1 | 2 | 16 |

Cómo leerla: un 1 es un acierto a la primera; un número alto, que la respuesta quedó muy abajo
entre las 183 entidades. Fíjate en qué tipos de arista (`r`) dan rangos bajos y en cuáles altos, y
en si el origen `h` tiene otras aristas que el modelo haya podido ver.

TransH y TransR tienen más piezas que ajustar que TransE (un plano o una matriz por tipo de
arista) y los mismos datos: en un grafo con casi todos los nodos de una a tres aristas, les
faltan ejemplos para ajustarlas. No es un fallo del entrenamiento: con el mismo código, en
*Nations* —un grafo de prueba pequeño pero denso— TransH empata con TransE
([`link_prediction.ipynb`](link_prediction.ipynb), sección 3).

## 5. La pregunta que nos importa: ¿qué parte?

Se esconde la arista de parte de un enunciado y se le pregunta al modelo cuál de las **dos**
partes encaja mejor. Con dos candidatas, el MRR no aporta nada que no diga el acierto (vale
`(1 + acierto) / 2`), así que se mide el **acierto**: la fracción de enunciados con la parte bien.

- Solo los **47 enunciados unilaterales**. Los 36 gemelos —la misma frase emitida una vez por
  parte, como en *«each party shall…»*— tienen las dos partes por construcción y no hay nada que
  adivinar.
- Se reparten los 47 en cinco grupos; se esconde un grupo, se entrena con el resto del grafo y se
  pregunta por ese grupo. Así hasta pasar por los cinco, y todo tres veces con repartos distintos.
- Dos referencias sin modelo: **siempre la parte más frecuente** y **la mayoría de su cláusula**
  (la parte que más aparece entre los otros enunciados de la misma cláusula).

In [9]:
queries = [q for q in lp.party_queries(SV) if q[0] not in SV.twin]
twins = [q for q in lp.party_queries(SV) if q[0] in SV.twin]
parties = sorted(SV.parties)
correct = collections.defaultdict(list)  # método -> [acierto de cada repetición]
predictions = collections.defaultdict(dict)  # método -> {enunciado: parte} de la primera repetición
start = time.time()
for rep in range(3):
    fold_of = lp.folds(len(queries), 5, seed=rep)
    hits = collections.Counter()
    for fold in range(5):
        test = [q for q, f in zip(queries, fold_of) if f == fold]
        known = [q for q, f in zip(queries, fold_of) if f != fold] + twins
        graph = [t for t in SV.triples if t not in set(test)]
        base = lp.t1_baselines(SV, known, test)
        guesses = {"siempre la más frecuente": base["mayoría"], "mayoría de su cláusula": base["cláusula"]}
        for m in MODELS:
            scores = lp.kge_tail_scores(lp.train_kge(m, graph, e2i, r2i, seed=rep, **CONFIG[m]), [(h, r) for h, r, _ in test], e2i, r2i)
            guesses[m] = {sid: parties[int(np.argmax([row[e2i[p]] for p in parties]))] for (sid, _, _), row in zip(test, scores)}
        for method, guess in guesses.items():
            hits[method] += sum(guess[sid] == p for sid, _, p in test)
            if rep == 0:
                predictions[method].update(guess)
    for method, k in hits.items():
        correct[method].append(k / len(queries))
print(f"{3 * 5 * len(MODELS)} entrenamientos en {time.time() - start:.0f} s")
lp.table(sorted([{"método": m, "acierto": float(np.mean(v)), "peor repetición": min(v), "mejor repetición": max(v)} for m, v in correct.items()], key=lambda r: -r["acierto"]))

45 entrenamientos en 143 s


| método | acierto | peor repetición | mejor repetición |
|---|---|---|---|
| mayoría de su cláusula | 0.858 | 0.830 | 0.872 |
| siempre la más frecuente | 0.745 | 0.745 | 0.745 |
| TransE | 0.660 | 0.596 | 0.766 |
| TransR | 0.574 | 0.447 | 0.766 |
| TransH | 0.553 | 0.489 | 0.638 |

En [`link_prediction.ipynb`](link_prediction.ipynb), con estos mismos modelos y repartos, ninguno
de los tres llegó a la mayoría de su cláusula, ni siquiera a «siempre la más frecuente», y TransH y
TransR, pensados para el muchos-a-uno, quedaron por debajo de TransE. Esta tabla debería repetirlo.

### Por qué

Al esconder la arista de parte, ¿qué le queda al modelo para decidir? Las otras aristas del
enunciado. Cuántas tiene cada uno de los 47:

In [10]:
others = collections.Counter(
    sum(1 for h, r, t in SV.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) for sid, _, _ in queries
)
lp.table([{"aristas además de la de parte": k, "enunciados": others[k]} for k in sorted(others)])

| aristas además de la de parte | enunciados |
|---|---|
| 0 | 2 |
| 1 | 20 |
| 2 | 14 |
| 3 | 9 |
| 4 | 1 |
| 5 | 1 |

La mayoría tiene una o dos: casi siempre `is_part_of` hacia su cláusula y, a veces, un término
definido que usa o una condición que cuelga de él. **Ninguna de esas aristas dice quién carga con
el enunciado.** Lo único que el modelo puede deducir es «se parece a los otros enunciados de su
cláusula», que es justo la mayoría de la cláusula, y eso hecho peor. Los que no tienen ninguna
otra arista quedan aislados: el modelo no tiene nada que mirar.

El muchos-a-uno era un problema real para TransE, pero no el que importa aquí. TransH y TransR lo
resuelven dejando que cada enunciado tenga su sitio lejos de la parte, y con eso tampoco hay de
dónde sacar la parte.

Lo que dice quién carga está **en el texto**, no en el grafo. Unos casos en que TransE falla:

In [11]:
fails = [(sid, p) for sid, _, p in queries if predictions["TransE"][sid] != p][:6]
lp.table([{
    "enunciado": sid,
    "texto": lp.statement_text(SV.statements[sid])[:80],
    "lo que ve el modelo": ", ".join(f"{r} {t if h == sid else h}" for h, r, t in SV.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) or "nada",
    "TransE dice": SV.parties[predictions["TransE"][sid]]["name"][:22],
    "correcta": SV.parties[p]["name"][:22],
} for sid, p in fails])

| enunciado | texto | lo que ve el modelo | TransE dice | correcta |
|---|---|---|---|---|
| obligation-3 | Marketing Affiliate shall pay to Equidata, as compensation for its providing of  | is_part_of clause-1, uses term-2, is_part_of reference-1, is_part_of value-1 | Equidata, Inc. | National Credit Report |
| obligation-4 | Such amounts shall be billed on a bi-monthly basis by Equidata | is_part_of clause-1 | National Credit Report | Equidata, Inc. |
| obligation-5 | Said amounts charged to Equidata will be billed separately to Marketing Affiliat | is_part_of clause-1 | National Credit Report | Equidata, Inc. |
| obligation-8 | Notice will be given to Marketing Affiliate in writing no less than 30 days prio | is_part_of condition-4, is_part_of clause-1, is_part_of value-2 | National Credit Report | Equidata, Inc. |
| obligation-12 | the Marketing Affiliate will be billed for those items in accordance with Equida | is_part_of clause-1, is_part_of reference-3 | National Credit Report | Equidata, Inc. |
| obligation-14 | The Marketing Affiliate agrees to pay a late charge of 1 l/2% per month on the u | is_part_of condition-9, is_part_of clause-1, is_part_of value-3 | Equidata, Inc. | National Credit Report |

## En resumen

- El grafo de SteelVault son 275 tripletas `(h, r, t)` entre 183 entidades.
- Los tres modelos aprenden algo del grafo en general (paso 4), TransE más que los otros dos.
- Para la parte de un enunciado (paso 5) no superan a mirar la mayoría de su cláusula. No es por
  el modelo: el grafo no contiene la información. Cada enunciado tiene una o dos aristas aparte
  de la de su parte, y ninguna dice quién carga con él.
- Corre en la GPU, y entrenar cada modelo lleva segundos.